# ASVspoof 2021 DF – dataset check

This notebook checks whether the ASVspoof 2021 DF files on disk match what is written in the official documentation: `README.DF.txt`, `keys/DF/README.txt`, the eval-package README and the main paper (Liu et al., 2023, arXiv version 3).

**How to use it**

- Run the cells from top to bottom.
- The notebook only **reads** the data – nothing in `datasets/` is changed. It reads only the headers of the audio files (`soundfile.info`), not the sound itself.
- Every check prints one line:
  - `[OK]` – the files match the documentation,
  - `[DIFF]` – the files do not match the documentation,
  - `[INFO]` – an observation, there is nothing to compare with.

**Output files**

In `outputs/03_asvspoof_2021_df_dataset_exploration_output/` (not committed to git):

- `audio_info.csv` – properties of every audio file (step 8a),
- `checks.csv` – table of all checks (step 9).

In `results/03_asvspoof_2021_df/03_asvspoof_2021_df_dataset_exploration_output/` (committed to git):

- `dataset_exploration_report.md` – report with the results of all steps (step 10).

### Setup – paths and settings

**What it does:** Loads the libraries, finds the project folder and sets the paths to the dataset (`datasets/03_asvspoof_2021_df`), to the outputs folder for working files and to the results folder for the report. The audio comes in four archives; each was unpacked into its own folder (`ASVspoof2021_DF_eval_part_00` … `_03`). It also defines the names of the three subsets, of the nine codecs, of the three source corpora and of the 13 columns of the label file.

**Why:** All the next cells use these paths and names. The paths are relative to the project folder, so the notebook works on any computer without changes.

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("pyproject.toml not found")


REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "datasets" / "03_asvspoof_2021_df"
PART_DIRS = [DATA_DIR / f"ASVspoof2021_DF_eval_part_0{i}" for i in range(4)]
FIRST_PART = PART_DIRS[0]
KEYS_DIR = DATA_DIR / "keys" / "DF"
NOTEBOOK_NAME = "03_asvspoof_2021_df_dataset_exploration"
OUTPUTS_DIR = REPO_ROOT / "outputs" / f"{NOTEBOOK_NAME}_output"
RESULTS_DIR = REPO_ROOT / "results" / "03_asvspoof_2021_df" / f"{NOTEBOOK_NAME}_output"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

METADATA_FILE = KEYS_DIR / "CM" / "trial_metadata.txt"
TRIAL_LIST_FILE = FIRST_PART / "ASVspoof2021.DF.cm.eval.trl.txt"
COLUMNS = [
    "speaker_id",
    "file_name",
    "codec",
    "source",
    "attack",
    "key",
    "trim",
    "subset",
    "vocoder",
    "col_10",
    "col_11",
    "col_12",
    "col_13",
]
SUBSETS = ["progress", "eval", "hidden"]
# values of the codec column in the order of the conditions DF-C1 ... DF-C9 (eval-package README)
CODECS = [
    "nocodec",
    "low_mp3",
    "high_mp3",
    "low_m4a",
    "high_m4a",
    "low_ogg",
    "high_ogg",
    "mp3m4a",
    "oggm4a",
]
SOURCES = ["asvspoof", "vcc2018", "vcc2020"]
VOCODERS = [
    "bonafide",
    "traditional_vocoder",
    "neural_vocoder_autoregressive",
    "neural_vocoder_nonautoregressive",
    "waveform_concatenation",
    "unknown",
]
EVAL_PACKAGE = "eval-package README (web)"

print("Data folder:", DATA_DIR.relative_to(REPO_ROOT), "| exists:", DATA_DIR.exists())
for part_dir in PART_DIRS:
    print(" ", part_dir.name, "| exists:", part_dir.exists())

### Check list – helper function

**What it does:** Defines `add_check`, a small function that compares an **expected** value (from the documentation) with a **measured** value (from the files on disk). It prints `[OK]` if they match, `[DIFF]` if they do not, and `[INFO]` if there is nothing to compare with (only an observation). Every result is saved in the list `checks`.

**Why:** All results are collected in one place, so at the end they can be shown in one table and written to the report.

In [ ]:
checks = []


def add_check(check: str, source: str, expected, measured) -> None:
    ok = None if expected is None else expected == measured
    checks.append(
        {
            "check": check,
            "source": source,
            "expected": expected,
            "measured": measured,
            "ok": ok,
        }
    )
    status = "INFO" if ok is None else ("OK" if ok else "DIFF")
    print(f"[{status}] {check}: expected={expected!r}, measured={measured!r}")

### Step 1 – Folder structure

**What it does:** Compares the files of the first audio part with the list in `README.DF.txt` (section 1). The README calls the licence file `LICENSE.txt`, but the archive contains `LICENSE.DF.txt` – this known difference is kept as a DIFF. It also checks that the keys package contains the label file described in `keys/DF/README.txt`; the `ASV` folder is marked there as optional.

**Why:** Confirms that the audio archives and the keys package were downloaded and unpacked completely.

In [ ]:
found_top = sorted(p.name for p in DATA_DIR.iterdir() if not p.name.startswith("."))
print("Data folder contains:", found_top)

expected_items = {"ASVspoof2021.DF.cm.eval.trl.txt", "flac", "README.DF.txt", "LICENSE.txt"}
found_items = {p.name for p in FIRST_PART.iterdir() if not p.name.startswith(".")}
add_check(
    "ASVspoof2021_DF_eval_part_00 folder contents",
    "README.DF.txt, section 1",
    expected_items,
    found_items,
)
add_check(
    "keys/DF: CM/trial_metadata.txt exists",
    "keys/DF/README.txt",
    True,
    METADATA_FILE.exists(),
)
add_check(
    "keys/DF: ASV folder exists",
    "keys/DF/README.txt (marked optional)",
    None,
    (KEYS_DIR / "ASV").exists(),
)

### Step 2 – Load the label files

**What it does:** Reads the label file `keys/DF/CM/trial_metadata.txt` into one table. Each row is one recording: speaker, file name, codec, source corpus, attack, real (`bonafide`) or fake (`spoof`), trimming, subset, vocoder type and four columns that are not documented (`col_10` … `col_13`). It also reads the list of evaluation files `ASVspoof2021.DF.cm.eval.trl.txt` from the first audio part.

**Why:** All the next checks use these tables. As in ASVspoof 2021 LA, the labels are not in the audio package – they come in the separate keys package.

In [ ]:
cm = pd.read_csv(
    METADATA_FILE,
    sep=" ",
    header=None,
    names=COLUMNS,
    dtype=str,
    keep_default_na=False,
)
trials = pd.read_csv(
    TRIAL_LIST_FILE,
    sep=" ",
    header=None,
    names=["file_name"],
    dtype=str,
    keep_default_na=False,
)
print("Rows in trial_metadata.txt:", len(cm))
print("Rows in the trial list:", len(trials))
display(cm)

### Step 3 – Format of the label files

**What it does:** Checks that every line of `trial_metadata.txt` has 13 columns, which values appear in the columns label, subset, trimming, source, codec and vocoder type, that the attacks from ASVspoof 2019 are A07–A19, that there are more than 100 different attacks, and that no file name appears twice.

**Why:** If the label file were broken, every later result would be wrong. The expected values come from the eval-package README (link 6 in section 3 of `docs/03_asvspoof_2021_df.md`) and the main paper (p. 5: *"in excess of 100 different spoofing attack algorithms"*).

In [ ]:
with open(METADATA_FILE, encoding="utf-8") as f:
    columns_per_line = {len(line.split()) for line in f if line.strip()}
add_check("trial_metadata.txt: columns per line", EVAL_PACKAGE, {13}, columns_per_line)

add_check("Values of key", EVAL_PACKAGE, {"bonafide", "spoof"}, set(cm["key"]))
add_check("Values of subset", EVAL_PACKAGE, set(SUBSETS), set(cm["subset"]))
add_check("Values of trim", EVAL_PACKAGE, {"notrim", "trim"}, set(cm["trim"]))
add_check("Values of source", EVAL_PACKAGE, set(SOURCES), set(cm["source"]))
add_check("Values of codec (DF-C1 ... DF-C9)", EVAL_PACKAGE, set(CODECS), set(cm["codec"]))
add_check("Values of vocoder", EVAL_PACKAGE, set(VOCODERS), set(cm["vocoder"]))

spoof = cm[cm["key"] == "spoof"]
add_check(
    "attack for spoof from ASVspoof 2019",
    EVAL_PACKAGE,
    {f"A{i:02d}" for i in range(7, 20)},
    set(spoof.loc[spoof["source"] == "asvspoof", "attack"]),
)
add_check(
    "More than 100 different attacks",
    "main paper, p. 5, section II-C",
    True,
    spoof["attack"].nunique() > 100,
)
add_check("Duplicated file names", "-", 0, int(cm["file_name"].duplicated().sum()))

### Step 4 – Number of recordings per subset

**What it does:** Counts real and fake recordings in the progress, eval and hidden subsets and compares the numbers with the main paper (p. 3, Table I). The paper does not give numbers for the hidden subset.

**Why:** Confirms that the dataset on disk is complete and the same as the one described in the paper.

In [ ]:
counts = pd.crosstab(cm["subset"], cm["key"]).reindex(SUBSETS)
counts["total"] = counts.sum(axis=1)
counts.loc["all"] = counts.sum()
display(counts)

expected_counts = {"progress": (5768, 53557), "eval": (14869, 519059), "hidden": None}
for subset, expected in expected_counts.items():
    measured = (int(counts.loc[subset, "bonafide"]), int(counts.loc[subset, "spoof"]))
    source = "main paper, p. 3, Table I" if expected else "not in the paper (observation)"
    add_check(f"{subset}: bona fide / spoof", source, expected, measured)

### Step 5 – Vocoder types per subset

**What it does:** Shows how many recordings of each vocoder type are in each subset. The paper groups the more than 100 attacks into five vocoder categories (p. 5); real recordings have the value `bonafide`.

**Why:** Shows which kinds of fakes each subset contains.

In [ ]:
vocoders = pd.crosstab(cm["vocoder"], cm["subset"]).reindex(
    index=VOCODERS, columns=SUBSETS, fill_value=0
)
display(vocoders)

add_check(
    "vocoder for bona fide",
    EVAL_PACKAGE,
    {"bonafide"},
    set(cm.loc[cm["key"] == "bonafide", "vocoder"]),
)

### Step 6 – Speakers

**What it does:** Counts the different speakers in each subset and in the two VCC source corpora, and compares the numbers with the main paper: Table I (p. 3) – 67 speakers (37 women, 30 men) in progress and 93 (50 women, 43 men) in eval; section II-C (p. 5) – *"12 in VCC 2018, 14 in VCC 2020"*. The label file has no information about gender, so gender is not checked.

**Why:** Confirms that all speakers of the dataset are present in the files.

In [ ]:
speakers = {subset: set(cm.loc[cm["subset"] == subset, "speaker_id"]) for subset in SUBSETS}
speaker_summary = pd.DataFrame(
    {"subset": SUBSETS, "speakers": [len(speakers[subset]) for subset in SUBSETS]}
).set_index("subset")
display(speaker_summary)

for subset, expected in {"progress": 67, "eval": 93}.items():
    add_check(
        f"{subset}: speakers",
        "main paper, p. 3, Table I",
        expected,
        len(speakers[subset]),
    )
add_check("hidden: speakers", "not in the paper (observation)", None, len(speakers["hidden"]))
for source, expected in {"vcc2018": 12, "vcc2020": 14}.items():
    add_check(
        f"{source}: speakers",
        "main paper, p. 5, section II-C",
        expected,
        cm.loc[cm["source"] == source, "speaker_id"].nunique(),
    )

### Step 6b – Codecs, sources and trimming

**What it does:** Counts the recordings for each codec (conditions DF-C1 to DF-C9) and for each source corpus in each subset. Expected (main paper): the conditions DF-C6 to DF-C9 appear only in eval (Table IV, p. 4, underlined); the progress subset contains only ASVspoof 2019 data (p. 8); the eval subset contains all three sources (p. 5); the hidden subset contains only ASVspoof 2019 data with the silence removed (p. 9).

**Why:** The codec, the source corpus and the silence are possible shortcuts in this dataset. The values in the codec column are matched to the conditions as in the eval-package README.

In [ ]:
codec_subset = pd.crosstab(cm["codec"], cm["subset"]).reindex(
    index=CODECS, columns=SUBSETS, fill_value=0
)
source_subset = pd.crosstab(cm["source"], cm["subset"]).reindex(
    index=SOURCES, columns=SUBSETS, fill_value=0
)
display(codec_subset)
display(source_subset)


def values_in(subset: str, column: str) -> set:
    return set(cm.loc[cm["subset"] == subset, column])


add_check(
    "progress: codecs (DF-C1 ... DF-C5)",
    "main paper, p. 4, Table IV",
    set(CODECS[:5]),
    values_in("progress", "codec"),
)
add_check(
    "eval: codecs (DF-C1 ... DF-C9)",
    "main paper, p. 4, Table IV",
    set(CODECS),
    values_in("eval", "codec"),
)
add_check(
    "progress: sources",
    "main paper, p. 8, section III-C",
    {"asvspoof"},
    values_in("progress", "source"),
)
add_check(
    "eval: sources", "main paper, p. 5, section II-C", set(SOURCES), values_in("eval", "source")
)
add_check(
    "hidden: sources",
    "main paper, p. 9, section IV-A",
    {"asvspoof"},
    values_in("hidden", "source"),
)

is_hidden = cm["subset"] == "hidden"
add_check(
    "hidden: all recordings trimmed (trim)",
    "main paper, p. 9, section IV-A; eval-package README (web)",
    True,
    bool((cm.loc[is_hidden, "trim"] == "trim").all()),
)
add_check(
    "progress and eval: no trimmed recordings (notrim)",
    "main paper, p. 9, section IV-A; eval-package README (web)",
    True,
    bool((cm.loc[~is_hidden, "trim"] == "notrim").all()),
)

### Step 7 – Audio files vs labels

**What it does:** Collects the audio files from the four `flac/` folders and compares them with the label file (labelled files missing on disk, audio files without a label, file names that appear in more than one folder). It also checks that the list of evaluation files (`ASVspoof2021.DF.cm.eval.trl.txt`) contains the same files, in the same order, as the label file.

**Why:** Every labelled recording must exist as a file, and every audio file must have a label.

In [ ]:
flac_paths = {}
files_per_folder = {}
duplicates_on_disk = 0
for part_dir in PART_DIRS:
    part_files = list((part_dir / "flac").glob("*.flac"))
    files_per_folder[part_dir.name] = len(part_files)
    for path in part_files:
        if path.stem in flac_paths:
            duplicates_on_disk += 1
        flac_paths[path.stem] = path

on_disk = set(flac_paths)
in_labels = set(cm["file_name"])
missing = in_labels - on_disk
extra = on_disk - in_labels

print(f"Audio files on disk = {len(on_disk)}, rows in the label file = {len(in_labels)}")
add_check("Labelled files missing on disk", "-", 0, len(missing))
add_check("Audio files without a label", "-", 0, len(extra))
add_check("File names in more than one audio folder", "-", 0, duplicates_on_disk)
add_check(
    "Trial list = file names in trial_metadata.txt (same order)",
    "README.DF.txt, section 1 (list of evaluation data)",
    True,
    trials["file_name"].tolist() == cm["file_name"].tolist(),
)

file_summary = pd.DataFrame(
    {"folder": list(files_per_folder), "flac_files": list(files_per_folder.values())}
).set_index("folder")
file_summary.loc["all"] = file_summary.sum()
display(file_summary)

### Step 8a – Read the audio file properties

**What it does:** Reads only the header of every audio file (not the sound itself): format, sample rate, bit depth, number of channels and length. The result is saved to `outputs/03_asvspoof_2021_df_dataset_exploration_output/audio_info.csv`; the next time the notebook loads this file instead of reading all recordings again. **The first run takes several minutes (611,829 files).**

**Why:** These properties are needed for the technical checks in the next cell and later for the table of recording features (silence, loudness, bandwidth, duration).

In [ ]:
INFO_CACHE = OUTPUTS_DIR / "audio_info.csv"

if INFO_CACHE.exists():
    info = pd.read_csv(INFO_CACHE, dtype={"file_name": str})
    print("Loaded from cache:", INFO_CACHE.relative_to(REPO_ROOT))
else:
    rows = []
    paths = sorted(flac_paths.values())
    print(f"Reading {len(paths)} files ...")
    for path in paths:
        meta = sf.info(path)
        rows.append(
            {
                "file_name": path.stem,
                "format": meta.format,
                "subtype": meta.subtype,
                "samplerate": meta.samplerate,
                "channels": meta.channels,
                "frames": meta.frames,
            }
        )
    info = pd.DataFrame(rows)
    info["duration_s"] = info["frames"] / info["samplerate"]
    info.to_csv(INFO_CACHE, index=False)
    print("Saved to:", INFO_CACHE.relative_to(REPO_ROOT))

audio = info.merge(cm, on="file_name", how="inner")
print("Audio files with labels:", len(audio))

### Step 8b – Check the audio file properties

**What it does:** Checks that all files are FLAC, 16 kHz and 16-bit (`README.DF.txt`, section 2; main paper, p. 2), shows the number of channels, and computes duration statistics for each subset, separately for real and fake recordings. The documentation gives no typical duration, so the median is only an observation.

**Why:** Confirms the technical description of the dataset – also for the files that come from VCC 2018 and VCC 2020. A clear difference in duration between real and fake recordings could be a shortcut that detectors use.

In [ ]:
add_check("File format", "README.DF.txt, section 2", {"FLAC"}, set(audio["format"]))
add_check(
    "Sample rate [Hz]",
    "README.DF.txt, section 2; main paper, p. 2",
    {16000},
    set(audio["samplerate"]),
)
add_check(
    "Bit depth (subtype)",
    "README.DF.txt, section 2; main paper, p. 2",
    {"PCM_16"},
    set(audio["subtype"]),
)
add_check("Number of channels", "not stated (observation)", None, set(audio["channels"]))

property_values = pd.DataFrame(
    {
        "property": ["format", "subtype", "samplerate", "channels"],
        "values found": [
            ", ".join(map(str, sorted(audio[col].unique())))
            for col in ["format", "subtype", "samplerate", "channels"]
        ],
    }
).set_index("property")
display(property_values)

duration_stats = audio.groupby(["subset", "key"])["duration_s"].describe()
duration_stats["count"] = duration_stats["count"].astype(int)
display(duration_stats.round(2))

median_duration = round(float(audio["duration_s"].median()), 2)
add_check("Median duration [s]", "not stated (observation)", None, median_duration)

### Step 8c – Duration histograms

**What it does:** Draws the same three charts as for ASVspoof 2019 LA and 2021 LA (density, number of recordings, percent), with real and fake recordings on top of each other. Only the progress and eval subsets are used (original recordings, without the hidden subset).

**Why:** Makes it easy to see at a glance whether real and fake recordings differ in length.

In [ ]:
notrim = audio[audio["trim"] == "notrim"]
max_duration = float(np.ceil(notrim["duration_s"].max()))
bins = np.linspace(0, max_duration, 101)


def plot_durations(data: pd.DataFrame, name: str) -> None:
    variants = [
        ("density", "Density", ""),
        ("counts", "Number of recordings", " (counts)"),
        ("percent", "Share of recordings in the group [%]", " (percent)"),
    ]
    for variant, y_label, title_end in variants:
        fig, ax = plt.subplots(figsize=(9, 4))
        for key, group in data.groupby("key"):
            weights = [100 / len(group)] * len(group) if variant == "percent" else None
            ax.hist(
                group["duration_s"],
                bins=bins,
                alpha=0.5,
                density=variant == "density",
                weights=weights,
                label=key,
            )
        ax.set_xlabel("Duration [s]")
        ax.set_ylabel(y_label)
        ax.set_title(f"ASVspoof 2021 DF, {name}: duration of bona fide vs spoof{title_end}")
        ax.legend()
        plt.show()


plot_durations(notrim, "progress + eval")

### Step 9 – Summary of all checks

**What it does:** Shows all checks in one table with their status (OK, DIFF, INFO), lists only the differences and saves the table to `outputs/03_asvspoof_2021_df_dataset_exploration_output/checks.csv`.

**Why:** One place to see what matches the documentation and what does not.

In [ ]:
summary = pd.DataFrame(checks)
summary["status"] = summary["ok"].map({True: "OK", False: "DIFF"}).fillna("INFO")
summary = summary[["status", "check", "expected", "measured", "source"]]
display(summary)

print("\nDifferences:")
display(summary[summary["status"] == "DIFF"])

summary.to_csv(OUTPUTS_DIR / "checks.csv", index=False)
print("Saved to:", OUTPUTS_DIR.relative_to(REPO_ROOT) / "checks.csv")

### Step 10 – Report

**What it does:** Writes the results of all steps (tables only, no charts) to `results/03_asvspoof_2021_df/03_asvspoof_2021_df_dataset_exploration_output/dataset_exploration_report.md`.

**Why:** The results can be read later without running the notebook again.

In [ ]:
REPORT_PATH = RESULTS_DIR / "dataset_exploration_report.md"


def fmt(value, max_items: int = 10) -> str:
    if isinstance(value, set):
        if not value:
            return "{} (empty)"
        items = sorted(map(str, value))
        if len(items) > max_items:
            items = [*items[:max_items], f"... (+{len(value) - max_items} more)"]
        return "{" + ", ".join(items) + "}"
    if isinstance(value, float):
        return f"{value:.2f}"
    return str(value)


def df_to_md(df: pd.DataFrame, index: bool = True) -> str:
    data = df.reset_index() if index else df
    header = "| " + " | ".join(map(str, data.columns)) + " |"
    separator = "|" + "---|" * len(data.columns)
    rows = ["| " + " | ".join(fmt(v) for v in row) + " |" for row in data.itertuples(index=False)]
    return "\n".join([header, separator, *rows])


status_counts = summary["status"].value_counts()
differences = summary[summary["status"] == "DIFF"]

sections = [
    ("Step 4 – Number of recordings per subset", counts),
    ("Step 5 – Number of recordings per vocoder type and subset", vocoders),
    ("Step 6 – Number of speakers per subset", speaker_summary),
    ("Step 6b – Number of recordings per codec and subset", codec_subset),
    ("Step 6b – Number of recordings per source and subset", source_subset),
    ("Step 7 – Audio files per folder", file_summary),
    ("Step 8 – Audio file properties", property_values),
    ("Step 8 – Duration of recordings [s]", duration_stats.round(2)),
]

lines = [
    "# ASVspoof 2021 DF – dataset check report",
    "",
    f"- Generated: {datetime.now():%Y-%m-%d %H:%M}",
    f"- Dataset folder: `{DATA_DIR.relative_to(REPO_ROOT)}`",
    f"- Notebook: `notebooks/{NOTEBOOK_NAME}.ipynb`",
    "",
    "## Overview",
    "",
    f"- Checks in total: {len(summary)}",
    f"- OK (matches the documentation): {status_counts.get('OK', 0)}",
    f"- DIFF (does not match): {status_counts.get('DIFF', 0)}",
    f"- INFO (observation, nothing to compare with): {status_counts.get('INFO', 0)}",
    "",
    "## Differences",
    "",
    df_to_md(differences, index=False) if len(differences) else "No differences found.",
    "",
    "## All checks",
    "",
    df_to_md(summary, index=False),
    "",
]
for title, table in sections:
    lines += [f"## {title}", "", df_to_md(table), ""]

REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
print("Report saved to:", REPORT_PATH.relative_to(REPO_ROOT))